In [ ]:
import numpy as np
import pandas as pd

import tensorflow as tf

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    LSTM,
    RepeatVector,
    TimeDistributed,
    Dense
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.preprocessing import StandardScaler

In [ ]:
if devices := tf.config.list_physical_devices("GPU"):
    print(f"Running on {devices}")
else:
    print("Running on CPU")

In [ ]:
# load API key
try:
    from google.colab import userdata
    IN_COLAB = True
except:
    IN_COLAB = False


if IN_COLAB:
    cdsapi_key = userdata.get("CDS-API-KEY")
else:
    import os

    %load_ext dotenv
    %dotenv

    cdsapi_key = os.getenv("CDS-API-KEY")


In [ ]:
# load dataset

import xarray as xr

# Geo-chunked surface levels data (optimised for time-series at a single location)
geochunked_sfc_url = "https://arco.datastores.ecmwf.int/cadl-arco-geo-002/arco/reanalysis_era5_single_levels/sfc/geoChunked.zarr"

# Open the zarr store with xarray, users must insert their API key where indicated.
ds = xr.open_zarr(
    geochunked_sfc_url,
    consolidated=True,
     storage_options = {
        "headers": {"Authorization": f"Bearer {cdsapi_key}"}
    }
)

# only use data from Frankfurt am Main
ds = ds.sel(latitude=50.7, longitude=8.41, method="nearest")
ds = ds.sel(time=slice("2020-01-01", None))

# select variables
# TODO: select features that mean something, I don't need all, longer timeframe is better. Don't use 
variables = ["t2m", "blh"]
ds = ds[variables]

# Inspect the variables
ds.load()
ds

In [ ]:
ds.isnull().sum()

In [ ]:
# plot raw example data

from earthkit import plots as ekp
from earthkit import transforms as ekt

import warnings
warnings.filterwarnings(
    "ignore",
    message="TimeSeries is an experimental new feature*"
)

# Select variable to plot
variable_name = "t2m"
time = slice("2016-07-01", "2026-07-1")

plot_data_hourly = ds[variable_name].sel(time=time) - 273.15 # transform from Kelvin to Celsius
plot_data_monthly = ekt.temporal.monthly_mean(plot_data_hourly)

chart = ekp.TimeSeries()

chart.line(plot_data_hourly, label="Hourly")
chart.line(plot_data_monthly, label="Monthly mean")

chart.title("2m Temperature at Frankfurt am Main")
chart.legend()

chart.show()

In [ ]:
# preprocess dataset
data = ds.to_dataarray(dim="features").transpose().values
scaler = StandardScaler()

data_scaled = scaler.fit_transform(data)
print(data)
print(data_scaled)

In [ ]:
# build sequences
LOOKBACK = 72      # previous 72 hours
HORIZON = 24       # predict next 24 hours

def create_sequences(data, lookback, horizon):
    X = []
    y = []

    for i in range(len(data) - lookback - horizon):
        X.append(data[i:i+lookback])
        y.append(data[i+lookback:i+lookback+horizon])

    return np.array(X), np.array(y)


X, y = create_sequences(
    data_scaled,
    LOOKBACK,
    HORIZON
)

print(X.shape)
print(y.shape)

In [ ]:
# train/test split
split = int(len(X) * 0.8)

X_train = X[:split]
X_test = X[split:]

y_train = y[:split]
y_test = y[split:]

In [ ]:
# define model
n_features = X.shape[2]

inputs = Input(shape=(LOOKBACK, n_features))

# Encoder
encoder = LSTM(
    128,
    activation="tanh"
)(inputs)

# Repeat context vector
decoder_input = RepeatVector(HORIZON)(encoder)

# Decoder
decoder = LSTM(
    128,
    activation="tanh",
    return_sequences=True
)(decoder_input)

outputs = TimeDistributed(
    Dense(n_features)
)(decoder)

model = Model(inputs, outputs)

model.summary()

In [ ]:
# compile model
model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

In [ ]:
print(np.isnan(X_train).any())
print(np.isnan(y_train).any())

print(np.max(X_train))
print(np.min(X_train))

In [ ]:
# train model

early_stop = EarlyStopping(
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=64,
    callbacks=[early_stop],
    shuffle=False
)

In [ ]:
# predict
pred = model.predict(X_test)

print(pred.shape)

In [ ]:
# rescale features
pred_original = scaler.inverse_transform(
    pred.reshape(-1, n_features)
).reshape(pred.shape)

y_original = scaler.inverse_transform(
    y_test.reshape(-1, n_features)
).reshape(y_test.shape)